# COO / PACD Cross-Verification — Local Test Notebook

Tests the full pipeline on a local machine **without** the production Docker stack.

| Component | Production | Local (this notebook) |
|---|---|---|
| LLM inference | Local VLLM server | Any OpenAI-compatible API |
| Vector store | Milvus server | **Milvus Lite** (file-based, no server) |
| Structured store | PostgreSQL | **In-memory dict** (or optional local Postgres) |
| Embeddings | Qwen3-VL-Embedding-2B | Same model, CPU-friendly |

## Pipeline steps covered
1. **PACD Ingestion** — load document → extract KV pairs via Vision LLM → index to Milvus Lite
2. **Template Setup** — load template image → extract field schema via Vision LLM
3. **COO Verification** — load COO → guided extraction → cross-reference vs PACD → generate report

## 0. Install Dependencies

In [ ]:
%pip install -q \
    milvus pymilvus \
    "langchain>=0.3" langchain-openai langchain-core \
    "langgraph>=0.2" \
    sentence-transformers \
    paddlepaddle paddleocr \
    pypdfium2 pillow \
    python-dotenv tabulate rich

## 1. Configuration

Set your API key and provider below. Any OpenAI-compatible endpoint works:
- **OpenAI**: `https://api.openai.com/v1`, model `gpt-4o-mini`
- **Groq**: `https://api.groq.com/openai/v1`, model `llama-3.3-70b-versatile` (vision: `meta-llama/llama-4-scout-17b-16e-instruct`)
- **Together AI**: `https://api.together.xyz/v1`
- **Google Gemini**: `https://generativelanguage.googleapis.com/v1beta/openai/`, model `gemini-2.0-flash`

In [ ]:
import os
from pathlib import Path

# ── LLM API ─────────────────────────────────────────────────────────────────
LLM_BASE_URL     = "https://api.openai.com/v1"   # change for other providers
LLM_API_KEY      = os.getenv("OPENAI_API_KEY", "YOUR_API_KEY_HERE")
LLM_TEXT_MODEL   = "gpt-4o-mini"                 # for extraction + report generation
LLM_VISION_MODEL = "gpt-4o-mini"                 # must support image inputs
LLM_TEMPERATURE  = 0.0
LLM_MAX_TOKENS   = 2048

# ── Milvus Lite (file-based — no server needed) ─────────────────────────────
MILVUS_DB_PATH   = "./coo_poc_milvus.db"          # local file, auto-created
PACD_COLLECTION  = "pacd_coo_documents"
DENSE_DIM        = 2048

# ── Sample document paths ───────────────────────────────────────────────────
# Point these at your actual PACD / COO / template files.
# Supported formats: PDF, PNG, JPG, TIFF
PACD_DOC_PATH     = Path("sample_pacd.pdf")       # <-- change this
TEMPLATE_DOC_PATH = Path("sample_template.png")   # <-- change this
COO_DOC_PATH      = Path("sample_coo.pdf")        # <-- change this

# ── Embedding model ─────────────────────────────────────────────────────────
EMBED_MODEL_NAME = "Qwen/Qwen3-VL-Embedding-2B"

print("✓ Config loaded")
print(f"  LLM endpoint : {LLM_BASE_URL}")
print(f"  Text model   : {LLM_TEXT_MODEL}")
print(f"  Vision model : {LLM_VISION_MODEL}")
print(f"  Milvus DB    : {MILVUS_DB_PATH}")

## 2. Imports

In [ ]:
import base64
import hashlib
import io
import json
import logging
import tempfile
import uuid
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

from pymilvus import (
    AnnSearchRequest,
    DataType,
    Function,
    FunctionType,
    MilvusClient,
    WeightedRanker,
)

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)   # suppress verbose SDK logs

print("✓ Imports OK")

## 3. Helper — Document Loading

Converts any supported file (PDF / PNG / JPG / TIFF) into a list of per-page JPEG `bytes` objects — the same format used by the production ingestion node.

In [ ]:
import pypdfium2 as pdfium

_IMG_EXTS = {".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp", ".gif", ".webp"}


def load_page_images(file_path: Path | str, dpi: int = 150) -> list[bytes]:
    """Return a list of JPEG-encoded bytes, one per page."""
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()
    pages_pil: list[Image.Image] = []

    if suffix == ".pdf":
        pdf = pdfium.PdfDocument(str(file_path))
        scale = dpi / 72
        for page in pdf:
            bitmap = page.render(scale=scale)
            pages_pil.append(bitmap.to_pil())
        pdf.close()
    elif suffix in _IMG_EXTS:
        img = Image.open(file_path)
        # Handle multi-frame (TIFF, GIF, etc.)
        try:
            while True:
                pages_pil.append(img.copy().convert("RGB"))
                img.seek(img.tell() + 1)
        except EOFError:
            pass
        if not pages_pil:
            pages_pil.append(img.convert("RGB"))
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

    jpeg_pages: list[bytes] = []
    for pil_img in pages_pil:
        buf = io.BytesIO()
        pil_img.convert("RGB").save(buf, format="JPEG", quality=85)
        jpeg_pages.append(buf.getvalue())

    return jpeg_pages


def load_page_images_from_bytes(raw_bytes: bytes, filename: str, dpi: int = 150) -> list[bytes]:
    """Same as load_page_images but from in-memory bytes."""
    suffix = Path(filename).suffix.lower()
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(raw_bytes)
        tmp_path = Path(tmp.name)
    try:
        return load_page_images(tmp_path, dpi=dpi)
    finally:
        tmp_path.unlink(missing_ok=True)


def make_doc_id(filename: str, raw_bytes: bytes) -> str:
    digest = hashlib.sha256(raw_bytes).hexdigest()[:12]
    stem = Path(filename).stem.lower().replace(" ", "_")
    return f"{stem}_{digest}"


def encode_image_b64(image_bytes: bytes, mime_type: str = "image/jpeg") -> str:
    """Encode image bytes as a base64 data URI for the Vision LLM."""
    b64 = base64.b64encode(image_bytes).decode("utf-8")
    return f"data:{mime_type};base64,{b64}"


print("✓ Document loading helpers ready")

## 4. Helper — Embedding Model + OCR

Loads the same `Qwen/Qwen3-VL-Embedding-2B` model used in production (runs on CPU, ~4 GB RAM). First run will download the model weights.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print(f"Loading embedding model: {EMBED_MODEL_NAME} ...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
print(f"✓ Embedding model loaded  (dim={embed_model.get_sentence_embedding_dimension()})")

In [ ]:
from paddleocr import PaddleOCR

print("Loading PaddleOCR ...")
ocr_engine = PaddleOCR(use_angle_cls=True, lang="en", show_log=False)
print("✓ OCR engine ready")


def embed_image(image_bytes: bytes) -> list[float]:
    """Compute a 2048-dim dense embedding from JPEG image bytes."""
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    vec = embed_model.encode(img, normalize_embeddings=True)
    return vec.tolist()


def ocr_image(image_bytes: bytes) -> str:
    """Run OCR and return all text from the image concatenated."""
    img_array = np.array(Image.open(io.BytesIO(image_bytes)).convert("RGB"))
    result = ocr_engine.ocr(img_array, cls=True)
    lines: list[str] = []
    if result and result[0]:
        for line in result[0]:
            text, confidence = line[1]
            if confidence > 0.5:
                lines.append(text)
    return " ".join(lines)


print("✓ embed_image() and ocr_image() ready")

## 5. LLM Factory

Creates LangChain `ChatOpenAI` clients pointing at the configured API endpoint.

In [ ]:
def get_text_llm() -> ChatOpenAI:
    return ChatOpenAI(
        base_url=LLM_BASE_URL,
        model=LLM_TEXT_MODEL,
        api_key=LLM_API_KEY,
        temperature=LLM_TEMPERATURE,
        max_tokens=LLM_MAX_TOKENS,
    )


def get_vision_llm() -> ChatOpenAI:
    return ChatOpenAI(
        base_url=LLM_BASE_URL,
        model=LLM_VISION_MODEL,
        api_key=LLM_API_KEY,
        temperature=LLM_TEMPERATURE,
        max_tokens=LLM_MAX_TOKENS,
    )


# Quick connectivity smoke-test
print("Testing LLM connection ...")
try:
    _test_llm = get_text_llm()
    _resp = _test_llm.invoke([HumanMessage(content="Reply with the single word: OK")])
    print(f"✓ LLM reachable — response: {_resp.content.strip()[:60]}")
except Exception as e:
    print(f"✗ LLM connection failed: {e}")
    print("  → Check LLM_BASE_URL and LLM_API_KEY above")

## 6. Milvus Lite — Collection Setup

Uses **Milvus Lite** (embedded, file-based) — no separate server process required.

In [ ]:
print(f"Connecting to Milvus Lite at: {MILVUS_DB_PATH}")
milvus_client = MilvusClient(uri=MILVUS_DB_PATH)
print("✓ Milvus Lite connected")


def setup_pacd_collection(drop_existing: bool = False) -> None:
    """Create (or recreate) the pacd_coo_documents collection."""
    if drop_existing and milvus_client.has_collection(PACD_COLLECTION):
        milvus_client.drop_collection(PACD_COLLECTION)
        print(f"  Dropped existing collection: {PACD_COLLECTION}")

    if milvus_client.has_collection(PACD_COLLECTION):
        count = milvus_client.query(PACD_COLLECTION, filter="", output_fields=["count(*)"])
        print(f"✓ Collection '{PACD_COLLECTION}' already exists — {count[0].get('count(*)', '?')} records")
        return

    schema = milvus_client.create_schema(auto_id=False, enable_dynamic_field=False)
    schema.add_field("id",             DataType.VARCHAR, is_primary=True, max_length=256)
    schema.add_field("user_id",        DataType.VARCHAR, max_length=128)
    schema.add_field("transaction_id", DataType.VARCHAR, max_length=128)
    schema.add_field("doc_category",   DataType.VARCHAR, max_length=16)
    schema.add_field("filename",       DataType.VARCHAR, max_length=512)
    schema.add_field("page_num",       DataType.INT32)
    schema.add_field("ocr_text",       DataType.VARCHAR, max_length=65535,
                     enable_analyzer=True, enable_match=True)
    schema.add_field("extracted_kv",   DataType.VARCHAR, max_length=65535)
    schema.add_field("dense",          DataType.FLOAT_VECTOR, dim=DENSE_DIM)
    schema.add_field("sparse",         DataType.SPARSE_FLOAT_VECTOR)

    # Auto-build BM25 sparse vectors from ocr_text
    bm25_fn = Function(
        name="ocr_to_sparse",
        function_type=FunctionType.BM25,
        input_field_names=["ocr_text"],
        output_field_names=["sparse"],
    )
    schema.add_function(bm25_fn)

    index_params = milvus_client.prepare_index_params()
    index_params.add_index(field_name="dense",  index_type="FLAT",                 metric_type="COSINE")
    index_params.add_index(field_name="sparse", index_type="SPARSE_INVERTED_INDEX", metric_type="BM25")
    index_params.add_index(field_name="user_id",        index_type="INVERTED")
    index_params.add_index(field_name="transaction_id", index_type="INVERTED")
    index_params.add_index(field_name="doc_category",   index_type="INVERTED")

    milvus_client.create_collection(
        PACD_COLLECTION,
        schema=schema,
        index_params=index_params,
    )
    print(f"✓ Created collection '{PACD_COLLECTION}'")


# Set drop_existing=True to start fresh on re-runs
setup_pacd_collection(drop_existing=False)

## 7. In-Memory Transaction Store

Replaces Postgres for local testing. All data lives in Python dicts within this notebook session.

In [ ]:
_transactions: dict[str, dict] = {}
_reports: dict[str, dict] = {}          # keyed by transaction_id
_template_attributes: dict[str, dict] = {}  # keyed by template_id


def create_transaction(user_id: str) -> dict:
    tx_id = str(uuid.uuid4())
    record = {"id": tx_id, "user_id": user_id, "status": "active",
              "created_at": datetime.now(timezone.utc).isoformat()}
    _transactions[tx_id] = record
    return record


def get_transaction(tx_id: str) -> dict | None:
    return _transactions.get(tx_id)


def save_report(tx_id: str, report: dict) -> None:
    _reports[tx_id] = report


def get_report(tx_id: str) -> dict | None:
    return _reports.get(tx_id)


def save_template_attributes(template_id: str, attributes: dict) -> None:
    _template_attributes[template_id] = attributes


def get_template_attributes(template_id: str) -> dict:
    return _template_attributes.get(template_id, {})


# ── Create a test transaction ────────────────────────────────────────────────
TEST_USER_ID = "user_test_001"
tx = create_transaction(TEST_USER_ID)
TRANSACTION_ID = tx["id"]

print(f"✓ Transaction created")
print(f"  user_id        : {TEST_USER_ID}")
print(f"  transaction_id : {TRANSACTION_ID}")

---
## Step 1 — PACD Document Ingestion

**Pipeline**: `Load file → Convert to page images → Vision LLM extracts KV pairs → Index to Milvus Lite`

Set `PACD_DOC_PATH` in the config cell (Section 1) to point at your PACD document.

### 1a. Load PACD Document → Page Images

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ── Load file (update PACD_DOC_PATH in config if needed) ─────────────────────
if not PACD_DOC_PATH.exists():
    raise FileNotFoundError(
        f"PACD file not found: {PACD_DOC_PATH}\n"
        "Set PACD_DOC_PATH in the config cell (Section 1)."
    )

pacd_raw_bytes = PACD_DOC_PATH.read_bytes()
pacd_doc_id    = make_doc_id(PACD_DOC_PATH.name, pacd_raw_bytes)
pacd_pages     = load_page_images(PACD_DOC_PATH)

print(f"✓ Loaded PACD: {PACD_DOC_PATH.name}")
print(f"  doc_id   : {pacd_doc_id}")
print(f"  pages    : {len(pacd_pages)}")

# Show first page thumbnail
fig, axes = plt.subplots(1, min(len(pacd_pages), 4), figsize=(16, 6))
if len(pacd_pages) == 1:
    axes = [axes]
for i, (ax, page_bytes) in enumerate(zip(axes, pacd_pages[:4])):
    ax.imshow(Image.open(io.BytesIO(page_bytes)))
    ax.set_title(f"PACD Page {i+1}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### 1b. Vision LLM — Extract Key-Value Pairs from PACD

In [ ]:
_PACD_SYSTEM_PROMPT = """\
You are a precise document data extraction assistant.
Examine the provided PACD (Pre-Arrival Customs Declaration) document image and extract ALL
key-value information visible on the page.
Include: names, dates, numbers, addresses, HS codes, quantities, amounts, reference numbers,
country of origin, importer/exporter details, and any labeled fields.

Respond ONLY with a JSON array. Each element must be:
{ "key": "<field_name_snake_case>", "value": "<extracted_value>", "confidence": <0.0-1.0> }

Rules:
- Use snake_case for key names (e.g. country_of_origin, invoice_number)
- If a value is unclear, include it with lower confidence (< 0.7)
- Do not include empty or null values
- Return an empty array [] if nothing can be extracted
"""

vision_llm = get_vision_llm()
pacd_kv_pairs: list[dict] = []

for page_idx, page_bytes in enumerate(pacd_pages):
    print(f"  Processing page {page_idx + 1}/{len(pacd_pages)} ...")
    image_url = encode_image_b64(page_bytes, "image/jpeg")

    msg = HumanMessage(content=[
        {"type": "text",  "text": _PACD_SYSTEM_PROMPT},
        {"type": "image_url", "image_url": {"url": image_url}},
        {"type": "text",  "text": "Extract all key-value pairs from this document page."},
    ])

    response = vision_llm.invoke([msg])
    raw = response.content.strip()

    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

    try:
        page_pairs = json.loads(raw)
        if not isinstance(page_pairs, list):
            page_pairs = []
    except json.JSONDecodeError:
        print(f"    Warning: could not parse JSON for page {page_idx + 1}")
        page_pairs = []

    # Annotate each pair with source page
    for pair in page_pairs:
        pair["page"] = page_idx

    pacd_kv_pairs.extend(page_pairs)
    print(f"    → {len(page_pairs)} KV pairs extracted")

print(f"\n✓ Total PACD KV pairs extracted: {len(pacd_kv_pairs)}")

In [ ]:
# Display extracted KV pairs as a table
from tabulate import tabulate

table_rows = [(p.get("key"), p.get("value"), f"{p.get('confidence', 0):.2f}", p.get("page", 0))
              for p in pacd_kv_pairs]
print(tabulate(table_rows, headers=["Key", "Value", "Confidence", "Page"], tablefmt="rounded_outline"))

### 1c. Index PACD Document into Milvus Lite

In [ ]:
print("Indexing PACD pages into Milvus Lite ...")

pacd_records = []

for page_idx, page_bytes in enumerate(pacd_pages):
    print(f"  Embedding + OCR for page {page_idx + 1}/{len(pacd_pages)} ...")

    # Dense embedding from image
    dense_vec = embed_image(page_bytes)

    # OCR text for BM25 sparse index
    ocr_text = ocr_image(page_bytes)
    if not ocr_text.strip():
        ocr_text = " ".join(p.get("value", "") for p in pacd_kv_pairs if p.get("page") == page_idx)

    # KV pairs for this page as JSON string
    page_kvs = [p for p in pacd_kv_pairs if p.get("page") == page_idx]
    extracted_kv_str = json.dumps([{"key": kv.get("key"), "value": kv.get("value")} for kv in page_kvs])

    record_id = f"{pacd_doc_id}_p{page_idx}"

    pacd_records.append({
        "id":             record_id,
        "user_id":        TEST_USER_ID,
        "transaction_id": TRANSACTION_ID,
        "doc_category":   "pacd",
        "filename":       PACD_DOC_PATH.name,
        "page_num":       page_idx,
        "ocr_text":       ocr_text[:65000],  # cap at Milvus varchar limit
        "extracted_kv":   extracted_kv_str[:65000],
        "dense":          dense_vec,
    })

# Insert all records
result = milvus_client.insert(PACD_COLLECTION, pacd_records)
print(f"\n✓ Indexed {result['insert_count']} PACD page(s) into Milvus Lite")
print(f"  Collection: {PACD_COLLECTION}")
print(f"  IDs: {[r['id'] for r in pacd_records]}")

---
## Step 2 — Template Setup

**Pipeline**: `Load template image → Vision LLM identifies fillable fields → Store field schema`

Set `TEMPLATE_DOC_PATH` in the config cell to point at your COO template image/PDF.

### 2a. Load Template Image

In [ ]:
if not TEMPLATE_DOC_PATH.exists():
    raise FileNotFoundError(
        f"Template file not found: {TEMPLATE_DOC_PATH}\n"
        "Set TEMPLATE_DOC_PATH in the config cell (Section 1)."
    )

template_raw_bytes = TEMPLATE_DOC_PATH.read_bytes()
template_id        = make_doc_id(TEMPLATE_DOC_PATH.name, template_raw_bytes)
template_pages     = load_page_images(TEMPLATE_DOC_PATH)
template_image     = template_pages[0]  # use first page as the template reference

print(f"✓ Loaded template: {TEMPLATE_DOC_PATH.name}")
print(f"  template_id  : {template_id}")
print(f"  pages loaded : {len(template_pages)}")

plt.figure(figsize=(10, 8))
plt.imshow(Image.open(io.BytesIO(template_image)))
plt.title(f"Template: {TEMPLATE_DOC_PATH.name}")
plt.axis("off")
plt.tight_layout()
plt.show()

### 2b. Vision LLM — Extract Template Field Schema

In [ ]:
_TEMPLATE_ATTR_SYSTEM_PROMPT = """\
You are a document template analyst.
Examine the provided COO (Certificate of Origin) template image and identify ALL labeled
fields, boxes, or sections that are meant to be filled in or printed on the final document.

Respond ONLY with a JSON object where:
- keys are field names in snake_case (e.g. "exporter_name", "invoice_number")
- values are the label exactly as seen on the template (e.g. "Exporter Name", "Invoice No.")

Example:
{
  "exporter_name": "Exporter Name",
  "country_of_origin": "Country of Origin",
  "invoice_number": "Invoice / Reference No."
}

Return an empty object {} if no labeled fields are found.
"""

print("Extracting template field schema via Vision LLM ...")
image_url = encode_image_b64(template_image, "image/jpeg")

msg = HumanMessage(content=[
    {"type": "text",      "text": _TEMPLATE_ATTR_SYSTEM_PROMPT},
    {"type": "image_url", "image_url": {"url": image_url}},
    {"type": "text",      "text": "Identify all labeled fields on this template."},
])

response = vision_llm.invoke([msg])
raw = response.content.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

try:
    template_attributes = json.loads(raw)
    if not isinstance(template_attributes, dict):
        template_attributes = {}
except json.JSONDecodeError:
    print("Warning: could not parse JSON — using empty attributes")
    template_attributes = {}

save_template_attributes(template_id, template_attributes)

print(f"\n✓ Extracted {len(template_attributes)} template fields:")
for k, v in template_attributes.items():
    print(f"  {k:35s}  →  {v}")

---
## Step 3 — COO Document Verification

**Pipeline**: 
1. Load COO document → page images  
2. Template retrieval from Milvus (or use template from Step 2 directly)  
3. Guided extraction — Vision LLM fills template fields from COO image  
4. Cross-reference — each COO field searched against PACD in Milvus  
5. Report generation — Text LLM writes verdict + narrative

### 3a. Load COO Document

In [ ]:
if not COO_DOC_PATH.exists():
    raise FileNotFoundError(
        f"COO file not found: {COO_DOC_PATH}\n"
        "Set COO_DOC_PATH in the config cell (Section 1)."
    )

coo_raw_bytes = COO_DOC_PATH.read_bytes()
coo_doc_id    = make_doc_id(COO_DOC_PATH.name, coo_raw_bytes)
coo_pages     = load_page_images(COO_DOC_PATH)
coo_image     = coo_pages[0]  # use first page

print(f"✓ Loaded COO: {COO_DOC_PATH.name}")
print(f"  doc_id : {coo_doc_id}")
print(f"  pages  : {len(coo_pages)}")

fig, axes = plt.subplots(1, min(len(coo_pages), 4), figsize=(16, 6))
if len(coo_pages) == 1:
    axes = [axes]
for i, (ax, page_bytes) in enumerate(zip(axes, coo_pages[:4])):
    ax.imshow(Image.open(io.BytesIO(page_bytes)))
    ax.set_title(f"COO Page {i+1}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3b. Template Retrieval (optional — find best-matching template from Milvus)

If you already have the right template from Step 2, skip this and set `confirmed_template_id = template_id` in the next cell.

In [ ]:
# Set to True to search Milvus for a matching template;
# Set to False to use the template from Step 2 directly.
SEARCH_FOR_TEMPLATE = False

if SEARCH_FOR_TEMPLATE:
    # The main 'templates' collection lives in the full Milvus server,
    # so for local testing we skip this and just use the template from Step 2.
    # If you have a local templates collection, connect to it here.
    print("⚠ Template retrieval against production Milvus requires the server.")
    print("  Falling back to template from Step 2.")
    confirmed_template_id   = template_id
    confirmed_template_name = TEMPLATE_DOC_PATH.name
    confirmed_template_attrs = get_template_attributes(template_id)
else:
    # Use the template extracted in Step 2 directly
    confirmed_template_id    = template_id
    confirmed_template_name  = TEMPLATE_DOC_PATH.name
    confirmed_template_attrs = get_template_attributes(template_id)

print(f"✓ Using template : {confirmed_template_name}")
print(f"  ID             : {confirmed_template_id}")
print(f"  Fields         : {list(confirmed_template_attrs.keys())}")

### 3c. Vision LLM — Extract COO Fields (guided by template schema)

In [ ]:
if confirmed_template_attrs:
    field_list_str = "\n".join(
        f'- {key}: "{label}"' for key, label in confirmed_template_attrs.items()
    )
    _COO_SYSTEM_PROMPT = f"""\
You are a precise document data extraction assistant.
Extract the following labeled fields from the COO (Certificate of Origin) document image.

Fields to extract:
{field_list_str}

Respond ONLY with a JSON object where keys are the field names and values are the
extracted string values. Use null for fields not found.

Example: {{ "exporter_name": "ABC Trading Co.", "country_of_origin": "Egypt" }}
"""
else:
    _COO_SYSTEM_PROMPT = """\
You are a precise document data extraction assistant.
Extract ALL labeled key-value information from this COO (Certificate of Origin) document image.

Respond ONLY with a JSON object where keys are field names in snake_case and values are the
extracted string values. Return {} if nothing can be extracted.
"""

print("Extracting COO fields via Vision LLM ...")
image_url = encode_image_b64(coo_image, "image/jpeg")

msg = HumanMessage(content=[
    {"type": "text",      "text": _COO_SYSTEM_PROMPT},
    {"type": "image_url", "image_url": {"url": image_url}},
    {"type": "text",      "text": "Extract the requested fields from this COO document."},
])

response = vision_llm.invoke([msg])
raw = response.content.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

try:
    coo_extracted_fields: dict[str, str] = json.loads(raw)
    if not isinstance(coo_extracted_fields, dict):
        coo_extracted_fields = {}
    # Remove None/null values
    coo_extracted_fields = {k: v for k, v in coo_extracted_fields.items() if v is not None}
except json.JSONDecodeError:
    print("Warning: could not parse JSON")
    coo_extracted_fields = {}

print(f"\n✓ Extracted {len(coo_extracted_fields)} COO fields:")
for k, v in coo_extracted_fields.items():
    print(f"  {k:35s}  →  {v}")

### 3d. Cross-Reference COO Fields Against PACD in Milvus Lite

In [ ]:
def search_pacd_for_field(
    field_key: str,
    coo_value: str,
    transaction_id: str,
    top_k: int = 3,
) -> list[dict]:
    """Search the PACD collection for records matching the given transaction_id.
    Returns the top_k most relevant records with their extracted_kv.
    """
    query_text = f"{field_key} {coo_value}"

    # Dummy dense vector — only BM25/sparse carries weight for text query
    dummy_dense = [0.0] * DENSE_DIM

    tx_filter = f'transaction_id == "{transaction_id}"'

    try:
        dense_req = AnnSearchRequest(
            data=[dummy_dense],
            anns_field="dense",
            param={"metric_type": "COSINE", "params": {}},
            limit=top_k,
            expr=tx_filter,
        )
        sparse_req = AnnSearchRequest(
            data=[query_text],
            anns_field="sparse",
            param={"metric_type": "BM25", "params": {}},
            limit=top_k,
            expr=tx_filter,
        )
        results = milvus_client.hybrid_search(
            PACD_COLLECTION,
            reqs=[dense_req, sparse_req],
            ranker=WeightedRanker(0.1, 0.9),  # heavily favour BM25 for text cross-reference
            limit=top_k,
            output_fields=["extracted_kv", "page_num", "filename"],
        )
        return results[0] if results else []
    except Exception as exc:
        print(f"    Milvus search error: {exc}")
        return []


def normalise(v: str) -> str:
    return str(v).strip().lower()


print("Cross-referencing COO fields against PACD ...\n")

cross_reference_results: list[dict] = []

for field_key, coo_value in coo_extracted_fields.items():
    hits = search_pacd_for_field(field_key, str(coo_value), TRANSACTION_ID)

    verdict = "not_found_in_pacd"
    pacd_value = None
    match_source_page = None

    if hits:
        top_hit = hits[0]
        entity = top_hit.get("entity", {})
        extracted_kv_raw = entity.get("extracted_kv", "[]")
        match_source_page = entity.get("page_num")

        try:
            kv_list = json.loads(extracted_kv_raw)
        except (json.JSONDecodeError, TypeError):
            kv_list = []

        # Find same key in PACD
        for kv in kv_list:
            if normalise(kv.get("key", "")) == normalise(field_key):
                pacd_value = kv.get("value")
                verdict = "match" if normalise(str(coo_value)) == normalise(str(pacd_value)) else "mismatch"
                break

    cross_reference_results.append({
        "field_key":   field_key,
        "coo_value":   str(coo_value),
        "pacd_value":  str(pacd_value) if pacd_value is not None else None,
        "verdict":     verdict,
        "source_page": match_source_page,
    })

    icon = {"match": "✅", "mismatch": "❌", "not_found_in_pacd": "⚠️"}.get(verdict, "?")
    print(f"  {icon} {field_key}: COO='{coo_value}'  PACD='{pacd_value}'  [{verdict}]")

print(f"\n✓ Cross-reference complete: {len(cross_reference_results)} fields checked")

### 3e. Generate Verification Report via Text LLM

In [ ]:
_REPORT_SYSTEM_PROMPT = """\
You are a trade compliance verification officer.
You have cross-referenced a Certificate of Origin (COO) document against PACD
(Pre-Arrival Customs Declaration) documents for the same transaction.

Below is the cross-reference result table. Write a professional verification report:
  1. A concise 2-3 sentence SUMMARY paragraph.
  2. A detailed NARRATIVE (3-5 paragraphs) describing findings, any mismatches,
     and what action the user should take.

Respond in plain text with two clearly labelled sections:
SUMMARY:
<summary text>

NARRATIVE:
<detailed narrative text>
"""

# Compute overall verdict
matched     = sum(1 for r in cross_reference_results if r["verdict"] == "match")
mismatched  = sum(1 for r in cross_reference_results if r["verdict"] == "mismatch")
not_found   = sum(1 for r in cross_reference_results if r["verdict"] == "not_found_in_pacd")
total       = len(cross_reference_results)

if total == 0 or not_found == total:
    overall_verdict = "INCONCLUSIVE"
elif mismatched > 0:
    overall_verdict = "FAIL"
else:
    overall_verdict = "PASS"

# Build table text for the prompt
table_header = "Field | COO Value | PACD Value | Verdict"
table_sep    = "-" * 60
table_lines  = [table_header, table_sep]
for r in cross_reference_results:
    table_lines.append(
        f"{r['field_key']} | {r['coo_value']} | {r['pacd_value'] or 'N/A'} | {r['verdict']}"
    )
table_text = "\n".join(table_lines)

user_prompt = (
    f"Overall preliminary verdict: {overall_verdict}\n"
    f"Matched: {matched}, Mismatched: {mismatched}, Not found in PACD: {not_found}\n\n"
    f"Cross-reference table:\n{table_text}\n\n"
    "Write the verification report."
)

print("Generating verification report via Text LLM ...")
text_llm = get_text_llm()
report_response = text_llm.invoke([
    SystemMessage(content=_REPORT_SYSTEM_PROMPT),
    HumanMessage(content=user_prompt),
])

report_text = report_response.content.strip()

# Parse SUMMARY / NARRATIVE sections
summary_text  = ""
narrative_text = ""
if "NARRATIVE:" in report_text:
    parts = report_text.split("NARRATIVE:", 1)
    narrative_text = parts[1].strip()
    if "SUMMARY:" in parts[0]:
        summary_text = parts[0].split("SUMMARY:", 1)[1].strip()
elif "SUMMARY:" in report_text:
    summary_text = report_text.split("SUMMARY:", 1)[1].strip()
else:
    narrative_text = report_text

# Assemble final report dict
verification_report = {
    "transaction_id":          TRANSACTION_ID,
    "coo_doc_id":               coo_doc_id,
    "confirmed_template_id":    confirmed_template_id,
    "confirmed_template_name":  confirmed_template_name,
    "overall_verdict":          overall_verdict,
    "discrepancy_table":        cross_reference_results,
    "summary":                  summary_text,
    "narrative":                narrative_text,
    "matched_count":            matched,
    "mismatched_count":         mismatched,
    "not_found_count":          not_found,
    "created_at":               datetime.now(timezone.utc).isoformat(),
}

save_report(TRANSACTION_ID, verification_report)
print(f"\n✓ Report generated and stored for transaction {TRANSACTION_ID}")

---
## Step 4 — Results Display

In [ ]:
# ── Verdict banner ───────────────────────────────────────────────────────────
from IPython.display import HTML, display

_VERDICT_COLORS = {"PASS": "#1e7e34", "FAIL": "#c82333", "INCONCLUSIVE": "#856404"}
_VERDICT_BG     = {"PASS": "#d4edda", "FAIL": "#f8d7da", "INCONCLUSIVE": "#fff3cd"}
_VERDICT_ICONS  = {"PASS": "✅", "FAIL": "❌", "INCONCLUSIVE": "⚠️"}

v = verification_report["overall_verdict"]
display(HTML(f"""
<div style="padding:16px; border-radius:8px; background:{_VERDICT_BG[v]};
            border:2px solid {_VERDICT_COLORS[v]}; margin:12px 0">
  <h2 style="color:{_VERDICT_COLORS[v]}; margin:0">
    {_VERDICT_ICONS[v]}&nbsp; Verification Verdict: {v}
  </h2>
  <p style="margin:8px 0 0 0">
    Matched: <b>{verification_report['matched_count']}</b> &nbsp;|
    Mismatched: <b>{verification_report['mismatched_count']}</b> &nbsp;|
    Not found in PACD: <b>{verification_report['not_found_count']}</b> &nbsp;|
    Total fields: <b>{len(cross_reference_results)}</b>
  </p>
</div>
"""))

In [ ]:
# ── Discrepancy table ────────────────────────────────────────────────────────
import pandas as pd

_VERDICT_ICON_MAP = {"match": "✅ match", "mismatch": "❌ mismatch", "not_found_in_pacd": "⚠️ not in PACD"}

df = pd.DataFrame(verification_report["discrepancy_table"])
if not df.empty:
    df["verdict"] = df["verdict"].map(_VERDICT_ICON_MAP).fillna(df["verdict"])
    df = df.rename(columns={
        "field_key":   "Field",
        "coo_value":   "COO Value",
        "pacd_value":  "PACD Value",
        "verdict":     "Verdict",
        "source_page": "PACD Page",
    })[["Field", "COO Value", "PACD Value", "Verdict", "PACD Page"]]
    display(df.style.set_properties(**{"text-align": "left"}).set_table_styles(
        [{"selector": "th", "props": [("text-align", "left"), ("background-color", "#f1f1f1")]}]
    ))
else:
    print("(no discrepancy data)")

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────────
print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(verification_report["summary"] or "(not generated)")

In [ ]:
# ── Narrative report ─────────────────────────────────────────────────────────
print("=" * 70)
print("NARRATIVE")
print("=" * 70)
print(verification_report["narrative"] or "(not generated)")

In [ ]:
# ── Full report dict (for debugging / export) ─────────────────────────────────
import pprint
print("Full verification_report dict:")
pprint.pprint({k: v for k, v in verification_report.items() if k != "discrepancy_table"}, width=100)

---
## Appendix A — Inspect Milvus Lite Contents

In [ ]:
# List all records in the PACD collection
all_records = milvus_client.query(
    PACD_COLLECTION,
    filter=f'transaction_id == "{TRANSACTION_ID}"',
    output_fields=["id", "user_id", "transaction_id", "doc_category", "filename", "page_num", "ocr_text"],
    limit=50,
)

print(f"Records in '{PACD_COLLECTION}' for transaction {TRANSACTION_ID}:")
for rec in all_records:
    print(f"  id={rec['id']:40s} | cat={rec['doc_category']:6s} | page={rec['page_num']} | ocr_len={len(rec.get('ocr_text',''))}")

## Appendix B — Cleanup

In [ ]:
# Run this cell to drop the collection and start fresh
# (useful during iterative testing)

CONFIRM_DROP = False  # set to True to actually drop

if CONFIRM_DROP:
    milvus_client.drop_collection(PACD_COLLECTION)
    print(f"✓ Dropped collection: {PACD_COLLECTION}")
    print("  Re-run the Milvus setup cell (Section 6) to recreate it.")
else:
    print("Set CONFIRM_DROP = True to drop the collection.")